In [76]:
import cv2
import numpy as np
import mediapipe as mp
import winsound
import pyttsx3
import threading
import time

fp=mp.solutions.face_mesh
fm_model=fp.FaceMesh(static_image_mode=False,
    max_num_faces=2,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,)

def compute_ear(eye_indices, landmarks):
    p1, p2, p3, p4, p5, p6 = [landmarks[i] for i in eye_indices]
    A = np.linalg.norm(np.array(p2) - np.array(p6))
    B = np.linalg.norm(np.array(p3) - np.array(p5))
    C = np.linalg.norm(np.array(p1) - np.array(p4))
    return (A + B) / (2.0 * C)
def compute_mar(landmarks):
    # Using MediaPipe 468 landmark format
    top_inner = np.array(landmarks[13])
    bottom_inner = np.array(landmarks[14])
    top_outer = np.array(landmarks[0])
    bottom_outer = np.array(landmarks[17])
    left = np.array(landmarks[61])
    right = np.array(landmarks[291])

    A = np.linalg.norm(top_inner - bottom_inner)
    B = np.linalg.norm(top_outer - bottom_outer)
    C = np.linalg.norm(left - right)

    mar = (A + B) / (2.0 * C)
    return mar

def speak(text):
    engine = pyttsx3.init()
    engine.say(text)
    engine.runAndWait()

def beep():
    winsound.Beep(1000, 500)


right_eye_indices = [33, 159, 158, 133, 153, 144]
left_eye_indices = [362, 386, 385, 263, 380, 374]

last_close_count = 0
last_yawn_count = 0

vid=cv2.VideoCapture(0)
while vid.isOpened():
    success, f = vid.read()
    if not success:
        break
    rgb=f[:,:,::-1]
    result=fm_model.process(rgb)
    if result.multi_face_landmarks:
        for face_landmarks in result.multi_face_landmarks:
            h, w = f.shape[:2]
            landmarks = [(int(p.x * w), int(p.y * h)) for p in face_landmarks.landmark]
            re=compute_ear(right_eye_indices,landmarks)
            le=compute_ear(left_eye_indices,landmarks)
            mouth=compute_mar(landmarks)
            if re<0.25 and le<0.25:
                cv2.putText(f, f'sleeping', (30, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 1)
                last_close_count+=1
                if last_close_count > 30:
                    winsound.Beep(1000, 500)
            else:
                last_close_count=0
            if mouth > 0.6:
    
                cv2.putText(f, f'yawn', (300, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 1)
                last_yawn_count+=1
                if last_close_count > 3:
                    speak("Don't sleep nigga",)
            else:
                last_yawn_time = 0
                
        
    cv2.imshow("a",f)
    if cv2.waitKey(100)==ord("q"):
        break
cv2.destroyAllWindows()
vid.release() 
    